In [1]:
import pandas as pd
import numpy as np
import logging

In [2]:
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 150)

In [3]:
import warnings
warnings.filterwarnings('ignore')

## Data

In [ ]:
df = pd.read_feather('df_features_for_model')

### columns for model

- Для начала возьмем аудио-признаки, длительность трека и год релиза
- Будем предсказывать главное настроение трека

In [5]:
def data_preprocessing(df_):

    df = df_.copy()

    df['release_date'] = pd.to_datetime(df['release_date'])
    df['release_year'] = df['release_date'].dt.year

    df['genre_cnt'] = df['genre'].apply(lambda x: len(x))
    df['genre'] = df['genre'].apply(lambda x: ', '.join(sorted(x)))

    df['instrument_cnt'] = df['instrument'].apply(lambda x: len(x))
    df['instrument'] = df['instrument'].apply(lambda x: ', '.join(sorted(x)))

    df.drop([
        'track_id',
        'artist_id',
        'album_id',
        'path',
        'mood/theme',
        'track_name',
        'release_date',
        'url',
        'mood_class',
        'situation_class',
        'for_summer',
        'for_children',
        'for_holiday',
        'for_sport',
        'for_christmas',
        'for_travel',
        'for_party',
        'is_sad_dark', 
        'is_energetic', 
        'is_positive', 
        'is_romantic', 
        'is_calm', 
        'is_deep_emotional'
    ], axis=1, inplace=True)

    return df

In [6]:
df_prep = data_preprocessing(df)

In [7]:
df_prep.columns

Index(['duration', 'genre', 'instrument', 'artist_name', 'album_name',
       '0_Absolute energy', '0_Autocorrelation', '0_Centroid',
       '0_ECDF Percentile Count_0', '0_ECDF Percentile_0', '0_Entropy',
       '0_Histogram mode', '0_Interquartile range', '0_Kurtosis', '0_Max',
       '0_Mean absolute diff', '0_Mean diff', '0_Median absolute deviation',
       '0_Median absolute diff', '0_Median diff', '0_Min',
       '0_Negative turning points', '0_Neighbourhood peaks',
       '0_Signal distance', '0_Skewness', '0_Slope', '0_Zero crossing rate',
       'spectral_centroid_mean', 'zcr_mean', 'rms_mean', 'bandwidth_mean',
       'flatness_mean', 'spectral_centroid_std', 'rolloff_std', 'zcr_std',
       'rms_std', 'bandwidth_std', 'flatness_std', 'tempo', 'mfcc_mean_1',
       'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4', 'mfcc_mean_5',
       'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8', 'mfcc_mean_9',
       'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12', 'mfcc_mean_13',
       'mfcc_st

In [8]:
only_audio_full = [
    '0_Autocorrelation', '0_Centroid', '0_ECDF Percentile Count_0',
    '0_ECDF Percentile_0', '0_Entropy', '0_Histogram mode',
    '0_Interquartile range', '0_Kurtosis', '0_Max', '0_Mean absolute diff',
    '0_Mean diff', '0_Median absolute deviation', '0_Median absolute diff',
    '0_Median diff', '0_Min', '0_Negative turning points',
    '0_Neighbourhood peaks', '0_Signal distance', '0_Skewness', '0_Slope',
    '0_Zero crossing rate', 'spectral_centroid_mean', 'zcr_mean',
    'rms_mean', 'bandwidth_mean', 'flatness_mean', 'spectral_centroid_std',
    'rolloff_std', 'zcr_std', 'rms_std', 'bandwidth_std', 'flatness_std',
    'tempo', 'mfcc_mean_1', 'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4',
    'mfcc_mean_5', 'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8',
    'mfcc_mean_9', 'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12',
    'mfcc_mean_13', 'mfcc_std_1', 'mfcc_std_2', 'mfcc_std_3', 'mfcc_std_4',
    'mfcc_std_5', 'mfcc_std_6', 'mfcc_std_7', 'mfcc_std_8', 'mfcc_std_9',
    'mfcc_std_10', 'mfcc_std_11', 'mfcc_std_12', 'mfcc_std_13',
    'chroma_mean_1', 'chroma_mean_2', 'chroma_mean_3', 'chroma_mean_4',
    'chroma_mean_5', 'chroma_mean_6', 'chroma_mean_7', 'chroma_mean_8',
    'chroma_mean_9', 'chroma_mean_10', 'chroma_mean_11', 'chroma_mean_12',
    'chroma_std_1', 'chroma_std_2', 'chroma_std_3', 'chroma_std_4',
    'chroma_std_5', 'chroma_std_6', 'chroma_std_7', 'chroma_std_8',
    'chroma_std_9', 'chroma_std_10', 'chroma_std_11', 'chroma_std_12'
]

only_librosa = [
    'spectral_centroid_mean', 'zcr_mean',
    'rms_mean', 'bandwidth_mean', 'flatness_mean', 'spectral_centroid_std',
    'rolloff_std', 'zcr_std', 'rms_std', 'bandwidth_std', 'flatness_std',
    'tempo', 'mfcc_mean_1', 'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4',
    'mfcc_mean_5', 'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8',
    'mfcc_mean_9', 'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12',
    'mfcc_mean_13', 'mfcc_std_1', 'mfcc_std_2', 'mfcc_std_3', 'mfcc_std_4',
    'mfcc_std_5', 'mfcc_std_6', 'mfcc_std_7', 'mfcc_std_8', 'mfcc_std_9',
    'mfcc_std_10', 'mfcc_std_11', 'mfcc_std_12', 'mfcc_std_13',
    'chroma_mean_1', 'chroma_mean_2', 'chroma_mean_3', 'chroma_mean_4',
    'chroma_mean_5', 'chroma_mean_6', 'chroma_mean_7', 'chroma_mean_8',
    'chroma_mean_9', 'chroma_mean_10', 'chroma_mean_11', 'chroma_mean_12',
    'chroma_std_1', 'chroma_std_2', 'chroma_std_3', 'chroma_std_4',
    'chroma_std_5', 'chroma_std_6', 'chroma_std_7', 'chroma_std_8',
    'chroma_std_9', 'chroma_std_10', 'chroma_std_11', 'chroma_std_12'
]

# визуально маленькая изменчивость и без std
only_audio_part = [
    '0_Autocorrelation', '0_Centroid',
    '0_ECDF Percentile_0', '0_Histogram mode',
    '0_Interquartile range', '0_Kurtosis', '0_Max', '0_Mean absolute diff',
    '0_Mean diff', '0_Median absolute deviation', '0_Median absolute diff',
    '0_Min', '0_Negative turning points',
    '0_Neighbourhood peaks', '0_Signal distance', '0_Skewness', '0_Slope',
    'spectral_centroid_mean', 'zcr_mean',
    'rms_mean', 'bandwidth_mean', 'flatness_mean', 'spectral_centroid_std',
    'rolloff_std', 'zcr_std', 'rms_std', 'bandwidth_std', 'flatness_std',
    'tempo', 'mfcc_mean_1', 'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4',
    'mfcc_mean_5', 'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8',
    'mfcc_mean_9', 'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12',
    'mfcc_mean_13', 'chroma_mean_1', 'chroma_mean_2', 'chroma_mean_3', 'chroma_mean_4',
    'chroma_mean_5', 'chroma_mean_6', 'chroma_mean_7', 'chroma_mean_8',
    'chroma_mean_9', 'chroma_mean_10', 'chroma_mean_11', 'chroma_mean_12',
]

meta_full = [
    'duration', 'genre', 'instrument', 'artist_name', 'album_name',
    'release_year', 'genre_cnt', 'instrument_cnt'
]

meta_num = [
    'duration','release_year', 'genre_cnt', 'instrument_cnt'
]

## Train_test split

In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, StratifiedKFold

1. Классы не сбалансированы: romantic и deep_emotional встречаются в 6 раз реже, чем самый частотный класс
2. К-fold берем стратифицированный, так как классы не сбалансированы

In [10]:
X, y = df_prep.drop('major_mood_class', axis=1).loc[:, only_audio_full], df['major_mood_class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
y.value_counts(normalize=True)

major_mood_class
positive          0.328527
calm              0.277254
energetic         0.157149
sad_dark          0.127347
romantic          0.058899
deep_emotional    0.050824
Name: proportion, dtype: float64

In [12]:
y_train.value_counts(normalize=True)

major_mood_class
positive          0.336019
calm              0.276781
energetic         0.155832
sad_dark          0.123787
romantic          0.058780
deep_emotional    0.048801
Name: proportion, dtype: float64

In [13]:
y_test.value_counts(normalize=True)

major_mood_class
positive          0.311045
calm              0.278359
energetic         0.160222
sad_dark          0.135655
romantic          0.059175
deep_emotional    0.055544
Name: proportion, dtype: float64

In [14]:
# кросс-валидация с сохранением пропорций классов

splitter = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

## Models

In [15]:
import optuna
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import cross_val_score
import time

In [16]:
def print_save_best_stats(study, name, model, X_train, X_test, y_train, y_test):
    
    print('--------------------------------------------')
    print(f'{name}_best_params: {study.best_params}')
    print('--------------------------------------------')    
    print(f'{name}_best_cv_f1: {np.round(study.best_value, 4)}')

    start_time = time.perf_counter()
    best_model = model(**study.best_params).fit(X_train, y_train)
    end_time = time.perf_counter()
    preds_test = best_model.predict(X_test)
    f1_test = f1_score(y_test, preds_test, average='macro')

    print('--------------------------------------------')
    print(f'{name}_fit_time: {np.round(end_time - start_time, 5)}')

    print('--------------------------------------------')
    print(f'{name}_f1_test: {np.round(f1_test, 4)}')
    
    print('--------------------------------------------')
    print(classification_report(y_test, preds_test))

    return name, study.best_value, f1_test, end_time - start_time

In [17]:
def append_result(df_metrics, model_res):
    df_metrics.loc[len(df_metrics)] = model_res

In [18]:
df_metrics = pd.DataFrame(
    columns=['model', 'val_score', 'test_score', 'time2fit']
)

### DecisionTreeClassifier

In [19]:
def objective_dt(trial):

    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_depth = trial.suggest_int('max_depth', 3, 16)
    min_samples_split = trial.suggest_int('min_samples_split', 50, 1000, 50)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 30, 500, 30)

    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        criterion = criterion,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        random_state=42
    )
    
    score = cross_val_score(
        model, X_train, y_train, cv=splitter, scoring='f1_macro', n_jobs=-1
    ).mean()

    return score

In [20]:
study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=100, timeout=7200, show_progress_bar=True)

[I 2026-03-10 20:20:27,181] A new study created in memory with name: no-name-e2ab1754-48ee-40bc-aa7d-91e9223022ab


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-10 20:20:29,974] Trial 0 finished with value: 0.19681290822042788 and parameters: {'criterion': 'entropy', 'max_depth': 9, 'min_samples_split': 450, 'min_samples_leaf': 60, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.19681290822042788.
[I 2026-03-10 20:20:31,886] Trial 1 finished with value: 0.18288105253751055 and parameters: {'criterion': 'gini', 'max_depth': 13, 'min_samples_split': 300, 'min_samples_leaf': 390, 'max_features': None, 'class_weight': None}. Best is trial 0 with value: 0.19681290822042788.
[I 2026-03-10 20:20:32,901] Trial 2 finished with value: 0.216524215725969 and parameters: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 300, 'min_samples_leaf': 420, 'max_features': None, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.216524215725969.
[I 2026-03-10 20:20:33,195] Trial 3 finished with value: 0.19875596265010587 and parameters: {'criterion': 'entropy', 'max_depth': 13, 'min_samples_split': 

In [21]:
metrics_dt = print_save_best_stats(study_dt, 'dt', DecisionTreeClassifier, X_train, X_test, y_train, y_test)

--------------------------------------------
dt_best_params: {'criterion': 'gini', 'max_depth': 14, 'min_samples_split': 750, 'min_samples_leaf': 180, 'max_features': None, 'class_weight': 'balanced'}
--------------------------------------------
dt_best_cv_f1: 0.2364
--------------------------------------------
dt_fit_time: 0.83579
--------------------------------------------
dt_f1_test: 0.2273
--------------------------------------------
                precision    recall  f1-score   support

          calm       0.48      0.24      0.32      1303
deep_emotional       0.11      0.37      0.16       260
     energetic       0.29      0.44      0.35       750
      positive       0.52      0.12      0.20      1456
      romantic       0.10      0.32      0.15       277
      sad_dark       0.17      0.21      0.19       635

      accuracy                           0.24      4681
     macro avg       0.28      0.28      0.23      4681
  weighted avg       0.38      0.24      0.25      

In [22]:
append_result(df_metrics, metrics_dt)

### RandomForestClassifier

In [23]:
def objective_rf(trial):

    n_estimators = trial.suggest_int('n_estimators', 50, 2000, 10)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_depth = trial.suggest_int('max_depth', 3, 16)
    min_samples_split = trial.suggest_int('min_samples_split', 50, 1000, 50)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 30, 500, 30)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    class_weight = trial.suggest_categorical('class_weight', ['balanced_subsample', 'balanced', None])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        criterion = criterion,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        random_state=42
    )
    
    score = cross_val_score(
        model, X_train, y_train, cv=splitter, scoring='f1_macro', n_jobs=-1
    ).mean()

    return score

In [24]:
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=100, timeout=7200, show_progress_bar=True)

[I 2026-03-10 20:21:51,846] A new study created in memory with name: no-name-0370e236-5628-42c2-bd0c-d2c55132f987


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-10 20:22:11,826] Trial 0 finished with value: 0.2598885049865209 and parameters: {'n_estimators': 370, 'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 450, 'min_samples_leaf': 180, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.2598885049865209.
[I 2026-03-10 20:22:57,506] Trial 1 finished with value: 0.2592893975745248 and parameters: {'n_estimators': 1370, 'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 50, 'min_samples_leaf': 480, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.2598885049865209.
[I 2026-03-10 20:23:14,600] Trial 2 finished with value: 0.263128792053328 and parameters: {'n_estimators': 450, 'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 200, 'min_samples_leaf': 330, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}. Best is trial 2 with value: 0.263128792053328.
[I 2026-03-10 20:24:16,271] Trial 3 finished with value: 0.2770063

In [25]:
metrics_rf = print_save_best_stats(study_rf, 'rf', RandomForestClassifier, X_train, X_test, y_train, y_test)

--------------------------------------------
rf_best_params: {'n_estimators': 1320, 'criterion': 'gini', 'max_depth': 14, 'min_samples_split': 50, 'min_samples_leaf': 30, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}
--------------------------------------------
rf_best_cv_f1: 0.3109
--------------------------------------------
rf_fit_time: 90.52723
--------------------------------------------
rf_f1_test: 0.3128
--------------------------------------------
                precision    recall  f1-score   support

          calm       0.45      0.47      0.46      1303
deep_emotional       0.16      0.19      0.18       260
     energetic       0.34      0.45      0.39       750
      positive       0.53      0.39      0.45      1456
      romantic       0.14      0.26      0.19       277
      sad_dark       0.26      0.18      0.22       635

      accuracy                           0.38      4681
     macro avg       0.32      0.33      0.31      4681
  weighted avg    

In [26]:
append_result(df_metrics, metrics_rf)

### Catboost

In [27]:
def objective_cb(trial):

    bootstrap_type = trial.suggest_categorical(
        'bootstrap_type',
        ['Bayesian', 'Bernoulli']
    )

    params = {
        'iterations': trial.suggest_int('iterations', 500, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1),
        'depth': trial.suggest_int('depth', 4, 8),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 30, 500, 30),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 50),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'bootstrap_type': bootstrap_type,
        'rsm': trial.suggest_float('rsm', 0.5, 1.0),
        'auto_class_weights': trial.suggest_categorical(
            'auto_class_weights', ['Balanced', 'SqrtBalanced', None]
        ),
        'random_seed': 42,
        'verbose': False
    }

    if bootstrap_type == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float(
            "bagging_temperature", 0, 10
        )
    else:
        params['subsample'] = trial.suggest_float(
            'subsample', 0.5, 1.0
        )

    model = CatBoostClassifier(**params)
    
    score = cross_val_score(
        model, X_train, y_train, cv=splitter, scoring='f1_macro', n_jobs=-1
    ).mean()

    return score

In [28]:
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=100, timeout=14400, show_progress_bar=True)

[I 2026-03-10 22:42:20,827] A new study created in memory with name: no-name-e0f44aef-4dfc-43a2-8187-f1672d86e080


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-10 22:44:41,995] Trial 0 finished with value: 0.32232761329138565 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 924, 'learning_rate': 0.04661580876123578, 'depth': 5, 'min_data_in_leaf': 300, 'l2_leaf_reg': 41.21550977998224, 'random_strength': 0.11171799303370644, 'rsm': 0.9337818109760142, 'auto_class_weights': 'Balanced', 'subsample': 0.9912621493662179}. Best is trial 0 with value: 0.32232761329138565.
[I 2026-03-10 22:47:32,235] Trial 1 finished with value: 0.33881021200386036 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 2112, 'learning_rate': 0.07364727101002362, 'depth': 4, 'min_data_in_leaf': 30, 'l2_leaf_reg': 14.736954210973197, 'random_strength': 8.843603448743604, 'rsm': 0.9590159819971089, 'auto_class_weights': 'Balanced', 'subsample': 0.9317511531228448}. Best is trial 1 with value: 0.33881021200386036.
[I 2026-03-10 22:57:06,181] Trial 2 finished with value: 0.24017925631125603 and parameters: {'bootstrap_type': 'Bayesian', 'it

In [29]:
metrics_cb = print_save_best_stats(study_cb, 'cb', CatBoostClassifier, X_train, X_test, y_train, y_test)

--------------------------------------------
cb_best_params: {'bootstrap_type': 'Bernoulli', 'iterations': 2503, 'learning_rate': 0.029997385129826228, 'depth': 6, 'min_data_in_leaf': 390, 'l2_leaf_reg': 10.328812003731068, 'random_strength': 1.8951729188762898, 'rsm': 0.5710162360081892, 'auto_class_weights': 'Balanced', 'subsample': 0.5912085695510947}
--------------------------------------------
cb_best_cv_f1: 0.3525
0:	learn: 1.7888820	total: 113ms	remaining: 4m 43s
1:	learn: 1.7854270	total: 177ms	remaining: 3m 40s
2:	learn: 1.7831043	total: 231ms	remaining: 3m 12s
3:	learn: 1.7797527	total: 280ms	remaining: 2m 54s
4:	learn: 1.7767097	total: 345ms	remaining: 2m 52s
5:	learn: 1.7735834	total: 396ms	remaining: 2m 44s
6:	learn: 1.7705111	total: 452ms	remaining: 2m 41s
7:	learn: 1.7682075	total: 512ms	remaining: 2m 39s
8:	learn: 1.7663185	total: 558ms	remaining: 2m 34s
9:	learn: 1.7638154	total: 605ms	remaining: 2m 30s
10:	learn: 1.7609443	total: 655ms	remaining: 2m 28s
11:	learn: 1.7

In [30]:
append_result(df_metrics, metrics_cb)

In [31]:
df_metrics.to_feather('df_metrics')
df_metrics

,model,val_score,test_score,time2fit
0,dt,0.236391,0.227349,0.835789
1,rf,0.310884,0.312827,90.527232
2,cb,0.352458,0.350530,132.136754
